# ⚖️ ElectioAnalytics — Comparaison des Modèles
**Objectif :** Comparer 5 algorithmes ML sur la cible binaire Version B  
**Cible :** DROITE (0) vs EXCEPTION (1) — features socio-éco uniquement  
**But :** Justifier le choix du modèle final pour la prédiction 2027

---

## ⚙️ 0 — Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

# Modèles
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier

# Validation et métriques
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics         import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    RocCurveDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

FINAL_PATH  = 'data/final'
MODELS_PATH = 'data/models'
os.makedirs(MODELS_PATH, exist_ok=True)

COLS_CIBLES = ['cible_binaire', 'cible_multiclasse', 'cible_transition_enc']

print('✅ Imports OK')

✅ Imports OK


---
## 📂 1 — Chargement (Version B uniquement)

In [2]:
train = pd.read_csv(f'{FINAL_PATH}/ml_train_2017_vB.csv', dtype={'code_geo': str})
test  = pd.read_csv(f'{FINAL_PATH}/ml_test_2022_vB.csv',  dtype={'code_geo': str})

features = [c for c in train.columns
            if (c.endswith('_scaled') or c.endswith('_enc'))
            and c not in COLS_CIBLES]

X_train = train[features].values
y_train = train['cible_binaire'].values
X_test  = test[features].values
y_test  = test['cible_binaire'].values

print(f'Features ({len(features)}) : {features}')
print(f'\nTrain — DROITE={(y_train==0).sum()} / EXCEPTION={(y_train==1).sum()}')
print(f'Test  — DROITE={(y_test==0).sum()} / EXCEPTION={(y_test==1).sum()}')

Features (5) : ['taux_faible_diplome_scaled', 'revenu_median_scaled', 'log_delinquance_scaled', 'log_population_scaled', 'pct_abstention_scaled']

Train — DROITE=598 / EXCEPTION=50
Test  — DROITE=470 / EXCEPTION=178


---
## 🔧 2 — Définition des modèles

In [3]:
# Tous les modèles avec class_weight='balanced' quand disponible
# pour compenser le déséquilibre DROITE/EXCEPTION

modeles = {
    'Régression Logistique': LogisticRegression(
        class_weight='balanced',
        C=0.1,
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'SVM': SVC(
        class_weight='balanced',
        kernel='rbf',
        C=1.0,
        probability=True,   # nécessaire pour AUC-ROC
        random_state=42
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=7,
        weights='distance',  # communes proches = plus de poids
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    )
}

print(f'✅ {len(modeles)} modèles définis :')
for nom in modeles:
    print(f'   - {nom}')

✅ 5 modèles définis :
   - Régression Logistique
   - Random Forest
   - SVM
   - KNN
   - Gradient Boosting


---
## 🤖 3 — Cross-Validation k=5 sur tous les modèles

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultats_cv = {}

for nom, model in modeles.items():
    print(f'⏳ {nom}...')
    cv_res = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring=['accuracy', 'f1', 'roc_auc'],
        return_train_score=True,
        n_jobs=-1
    )
    resultats_cv[nom] = cv_res
    val_acc = cv_res['test_accuracy'].mean()
    val_f1  = cv_res['test_f1'].mean()
    val_auc = cv_res['test_roc_auc'].mean()
    print(f'   ✅ Acc={val_acc:.3f}  F1={val_f1:.3f}  AUC={val_auc:.3f}')

print('\n✅ Cross-validation terminée')

⏳ Régression Logistique...
   ✅ Acc=0.806  F1=0.387  AUC=0.846
⏳ Random Forest...
   ✅ Acc=0.929  F1=0.349  AUC=0.837
⏳ SVM...
   ✅ Acc=0.840  F1=0.438  AUC=0.889
⏳ KNN...
   ✅ Acc=0.927  F1=0.338  AUC=0.820
⏳ Gradient Boosting...
   ✅ Acc=0.906  F1=0.275  AUC=0.845

✅ Cross-validation terminée


In [5]:
# Tableau récapitulatif CV
rows = []
for nom, cv_res in resultats_cv.items():
    rows.append({
        'Modèle':          nom,
        'CV Accuracy':     f"{cv_res['test_accuracy'].mean():.3f} ± {cv_res['test_accuracy'].std():.3f}",
        'CV F1':           f"{cv_res['test_f1'].mean():.3f} ± {cv_res['test_f1'].std():.3f}",
        'CV AUC-ROC':      f"{cv_res['test_roc_auc'].mean():.3f} ± {cv_res['test_roc_auc'].std():.3f}",
        'Overfit Acc':     '⚠️' if (cv_res['train_accuracy'].mean() -
                                      cv_res['test_accuracy'].mean()) > 0.1 else '✅'
    })

df_cv = pd.DataFrame(rows).set_index('Modèle')
print('📊 Résultats Cross-Validation (train 2017, k=5) :')
display(df_cv)

📊 Résultats Cross-Validation (train 2017, k=5) :


,CV Accuracy,CV F1,CV AUC-ROC,Overfit Acc
Modèle,,,,
Régression Logistique,0.806 ± 0.012,0.387 ± 0.032,0.846 ± 0.064,✅
Random Forest,0.929 ± 0.006,0.349 ± 0.102,0.837 ± 0.066,✅
SVM,0.840 ± 0.030,0.438 ± 0.064,0.889 ± 0.041,✅
KNN,0.927 ± 0.010,0.338 ± 0.054,0.820 ± 0.086,✅
Gradient Boosting,0.906 ± 0.017,0.275 ± 0.058,0.845 ± 0.060,✅


---
## 🧪 4 — Évaluation sur le test 2022

In [6]:
resultats_test = {}

for nom, model in modeles.items():
    model.fit(X_train, y_train)
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    resultats_test[nom] = {
        'model':        model,
        'y_pred':       y_pred,
        'y_pred_proba': y_pred_proba,
        'acc':          accuracy_score(y_test, y_pred),
        'f1':           f1_score(y_test, y_pred),
        'auc':          roc_auc_score(y_test, y_pred_proba),
        'cm':           confusion_matrix(y_test, y_pred)
    }

    joblib.dump(model, f'{MODELS_PATH}/model_{nom.lower().replace(" ", "_")}_vB.pkl')

# Tableau comparatif test
rows_test = []
for nom, r in resultats_test.items():
    cv_acc = resultats_cv[nom]['test_accuracy'].mean()
    rows_test.append({
        'Modèle':          nom,
        'CV Acc (train)':  f"{cv_acc:.3f}",
        'Accuracy (test)': f"{r['acc']:.3f}",
        'F1 (test)':       f"{r['f1']:.3f}",
        'AUC-ROC (test)':  f"{r['auc']:.3f}",
    })

df_test = pd.DataFrame(rows_test).set_index('Modèle')
print('📊 Résultats sur test 2022 :')
display(df_test)

# Meilleur modèle
best_nom = max(resultats_test, key=lambda x: resultats_test[x]['auc'])
print(f'\n🏆 Meilleur modèle (AUC-ROC) : {best_nom} — AUC={resultats_test[best_nom]["auc"]:.3f}')

📊 Résultats sur test 2022 :


,CV Acc (train),Accuracy (test),F1 (test),AUC-ROC (test)
Modèle,,,,
Régression Logistique,0.806,0.887,0.787,0.884
Random Forest,0.929,0.736,0.086,0.845
SVM,0.840,0.832,0.625,0.890
KNN,0.927,0.762,0.238,0.773
Gradient Boosting,0.906,0.761,0.265,0.792



🏆 Meilleur modèle (AUC-ROC) : SVM — AUC=0.890


---
## 🟦 5 — Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (nom, r) in enumerate(resultats_test.items()):
    ax  = axes[i]
    cm  = r['cm']
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    annot  = np.array([
        [f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)" for j in range(2)]
        for i in range(2)
    ])
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=['DROITE', 'EXCEPTION'],
                yticklabels=['DROITE', 'EXCEPTION'],
                linewidths=1.5, cbar=False,
                annot_kws={'size': 11})
    ax.set_title(
        f'{nom}\nAcc={r["acc"]:.1%}  F1={r["f1"]:.3f}  AUC={r["auc"]:.3f}',
        fontweight='bold', fontsize=10
    )
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

axes[-1].set_visible(False)
plt.suptitle('Matrices de Confusion — Comparaison des 5 modèles (Test 2022, Version B)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/comparaison_confusion.png', dpi=120)
plt.show()

---
## 📈 6 — Courbes ROC superposées

In [ ]:
couleurs = {
    'Régression Logistique': '#e74c3c',
    'Random Forest':         '#2ecc71',
    'SVM':                   '#3498db',
    'KNN':                   '#9b59b6',
    'Gradient Boosting':     '#f39c12'
}

fig, ax = plt.subplots(figsize=(9, 7))

for nom, r in resultats_test.items():
    RocCurveDisplay.from_predictions(
        y_test, r['y_pred_proba'],
        name=f"{nom} (AUC={r['auc']:.3f})",
        color=couleurs[nom],
        ax=ax
    )

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Modèle aléatoire (AUC=0.500)')
ax.set_title('Courbes ROC — Comparaison des 5 modèles\nVersion B (socio-éco pur)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/comparaison_roc.png', dpi=120)
plt.show()

---
## 🕸️ 7 — Graphique Radar (vue globale)

In [ ]:
# Radar : Accuracy / F1 / AUC / CV Accuracy / (1 - Overfit)
categories = ['Accuracy', 'F1 Score', 'AUC-ROC', 'CV Accuracy', 'Stabilité CV']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for nom, r in resultats_test.items():
    cv_acc   = resultats_cv[nom]['test_accuracy'].mean()
    cv_std   = resultats_cv[nom]['test_accuracy'].std()
    stabilite = max(0, 1 - cv_std * 5)  # pénalise la variance

    valeurs = [
        r['acc'],
        r['f1'],
        r['auc'],
        cv_acc,
        stabilite
    ]
    valeurs += valeurs[:1]

    ax.plot(angles, valeurs, linewidth=2,
            linestyle='solid', label=nom, color=couleurs[nom])
    ax.fill(angles, valeurs, alpha=0.08, color=couleurs[nom])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('Comparaison globale des modèles\n(Version B — socio-éco pur)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/comparaison_radar.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 📊 8 — Tableau Final de Comparaison

In [ ]:
rows_final = []
for nom, r in resultats_test.items():
    cv_acc = resultats_cv[nom]['test_accuracy'].mean()
    cv_std = resultats_cv[nom]['test_accuracy'].std()
    overfit = resultats_cv[nom]['train_accuracy'].mean() - cv_acc

    rows_final.append({
        'Modèle':           nom,
        'CV Acc ± std':     f"{cv_acc:.3f} ± {cv_std:.3f}",
        'Test Accuracy':    round(r['acc'], 3),
        'Test F1':          round(r['f1'],  3),
        'Test AUC-ROC':     round(r['auc'], 3),
        'Overfit':          f"{overfit:+.3f}",
        'Overfit?':         '⚠️' if overfit > 0.1 else '✅'
    })

df_final = pd.DataFrame(rows_final).set_index('Modèle')

# Surligner le meilleur par métrique
print('📊 Tableau final de comparaison :')
display(df_final)

# Meilleurs par métrique
print('\n🏆 Meilleur par métrique :')
for metric in ['Test Accuracy', 'Test F1', 'Test AUC-ROC']:
    best = df_final[metric].idxmax()
    val  = df_final.loc[best, metric]
    print(f'   {metric:<20} → {best:<25} ({val})')

---
## 🎯 9 — Barplot comparatif

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
noms   = list(resultats_test.keys())
x      = np.arange(len(noms))
colors = [couleurs[n] for n in noms]

metriques = [
    ('Accuracy',  [resultats_test[n]['acc'] for n in noms], 'Test Accuracy'),
    ('F1 Score',  [resultats_test[n]['f1']  for n in noms], 'Test F1 Score'),
    ('AUC-ROC',   [resultats_test[n]['auc'] for n in noms], 'Test AUC-ROC'),
]

for ax, (titre, vals, ylabel) in zip(axes, metriques):
    bars = ax.bar(x, vals, color=colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' ', '\n') for n in noms], fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_title(titre, fontweight='bold')
    ax.set_ylabel(ylabel)
    # Marquer le meilleur
    best_idx = np.argmax(vals)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.suptitle('Comparaison des 5 modèles — Version B (socio-éco pur)\n'
             '🥇 = meilleur par métrique',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/comparaison_barplot.png', dpi=120)
plt.show()

---
## 📋 10 — Bilan et Choix du Modèle Final

In [ ]:
best_auc = max(resultats_test, key=lambda x: resultats_test[x]['auc'])
best_f1  = max(resultats_test, key=lambda x: resultats_test[x]['f1'])
best_acc = max(resultats_test, key=lambda x: resultats_test[x]['acc'])

print('=' * 65)
print('📋 BILAN COMPARAISON — 5 MODÈLES')
print('=' * 65)
display(df_final)

print(f"""
🏆 Meilleurs par métrique
   Accuracy  → {best_acc} ({resultats_test[best_acc]['acc']:.3f})
   F1 Score  → {best_f1}  ({resultats_test[best_f1]['f1']:.3f})
   AUC-ROC   → {best_auc} ({resultats_test[best_auc]['auc']:.3f})

📌 Critère de choix pour ElectioAnalytics
   AUC-ROC est la métrique principale car :
   → Mesure la capacité à ORDONNER les communes par risque de bascule
   → Robuste au déséquilibre des classes (DROITE=72% / EXCEPTION=28%)
   → Utilisable pour construire une carte de probabilité 2027

✅ Modèle retenu pour la prédiction 2027 : {best_auc}
   AUC-ROC = {resultats_test[best_auc]['auc']:.3f}
   Accuracy = {resultats_test[best_auc]['acc']:.3f}
   F1       = {resultats_test[best_auc]['f1']:.3f}

Fichiers sauvegardés
   data/models/comparaison_confusion.png
   data/models/comparaison_roc.png
   data/models/comparaison_radar.png
   data/models/comparaison_barplot.png
   data/models/model_*_vB.pkl  (5 modèles)

🚀 Prochaine étape : visualisations + scénarios 2027
""")